# 06 – Normalize Product Images

Normalize supplier product images into a consistent, reusable format before
Shopify export and synchronization.

## Responsibility

This notebook:

- loads the normalized product image references created in the transformation step
- validates that required image data is available
- checks source image dimensions and format
- normalizes images when supplier files exceed the target requirements
- prepares image metadata for later ecommerce export and synchronization
- keeps image processing independent from Shopify API mutations

## Input

- `valid_product_images.csv`

## Output

- normalized image files ready for ecommerce use
- image metadata that can be consumed by the Shopify export step

The image-processing logic is intentionally separated from Shopify-specific API
operations so it can be reused for future suppliers with similar image issues.

## Setup

Define reusable paths and image-processing settings in one place so the notebook
can be adapted to another supplier without changing processing logic throughout
the notebook.

In [ ]:
from pathlib import Path
from io import BytesIO

import pandas as pd
import requests
from PIL import Image, ImageChops

import re
from urllib.parse import urlparse

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


SUPPLIER = "snickers"

DATA_DIR = Path("../data") / SUPPLIER
IMAGE_DATA_PATH = DATA_DIR / "valid_product_images.csv"

OUTPUT_DIR = Path("../data/shopify/normalized_images")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANVAS_SIZE = 1600
MARGIN = 140
JPEG_QUALITY = 95

In [ ]:
retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(max_retries=retry_strategy),
)

## Load Image Data

Load the validated image references produced by the transformation pipeline.
The notebook stops early if the expected input file is missing.

In [ ]:
if not IMAGE_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Image data file not found: {IMAGE_DATA_PATH}"
    )

product_images = pd.read_csv(
    IMAGE_DATA_PATH,
    dtype={
        "product_id": "string",
        "variant_sku": "string",
    },
)

print(f"Loaded {len(product_images):,} image records.")
product_images.head()

In [ ]:
# Count image relationships separately from unique source images.
image_relationships = len(product_images)
unique_image_urls = product_images["image_url"].nunique()

print(f"Image relationships: {image_relationships:,}")
print(f"Unique source images: {unique_image_urls:,}")

## Prepare Unique Source Images

A supplier image can be referenced by multiple product variants. Processing the
same source image repeatedly would waste network and processing resources.

The image relationships are therefore preserved separately, while each unique
source image is normalized only once.

In [ ]:
# Create one processing record per unique source image.
unique_images = (
    product_images[["image_url"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

assert len(unique_images) == product_images["image_url"].nunique()

print(f"Images to process: {len(unique_images):,}")

## Validate Image References

Validate the image dataset before downloading or transforming files. Invalid or
missing URLs are identified early so image-processing failures are easier to
trace back to the source data.

In [ ]:
missing_image_urls = product_images["image_url"].isna().sum()

blank_image_urls = (
    product_images["image_url"]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

assert missing_image_urls == 0, (
    f"Missing image URLs found: {missing_image_urls}"
)

assert blank_image_urls == 0, (
    f"Blank image URLs found: {blank_image_urls}"
)

print("Image URL validation passed.")

## Normalize Product Image

Supplier images can have inconsistent whitespace, dimensions and product scale.

The normalization function:

- converts the image to RGB
- detects the visible product area against a white background
- removes unnecessary surrounding whitespace
- scales the product proportionally
- centers it on a consistent square canvas

The original aspect ratio is preserved.

In [ ]:
def normalize_product_image(img):
    img = img.convert("RGB")

    # Detect the non-white product area.
    background = Image.new("RGB", img.size, "white")
    diff = ImageChops.difference(img, background)

    # Ignore very small differences from pure white.
    diff = diff.convert("L").point(
        lambda p: 0 if p < 15 else 255
    )

    bbox = diff.getbbox()

    cropped = img if bbox is None else img.crop(bbox)

    max_product_size = CANVAS_SIZE - (2 * MARGIN)

    scale = min(
        max_product_size / cropped.width,
        max_product_size / cropped.height,
    )

    new_size = (
        round(cropped.width * scale),
        round(cropped.height * scale),
    )

    resized = cropped.resize(
        new_size,
        Image.Resampling.LANCZOS,
    )

    canvas = Image.new(
        "RGB",
        (CANVAS_SIZE, CANVAS_SIZE),
        "white",
    )

    x = (CANVAS_SIZE - resized.width) // 2
    y = (CANVAS_SIZE - resized.height) // 2

    canvas.paste(resized, (x, y))

    return canvas

In [ ]:
def build_normalized_filename(image_url):
    # Keep the supplier filename recognizable while making it filesystem-safe.
    original_name = Path(
        urlparse(image_url).path
    ).name

    original_name = re.sub(
        r"\.(jpg|jpeg|png|webp)$",
        "",
        original_name,
        flags=re.IGNORECASE,
    )

    safe_name = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        original_name,
    )

    return f"{safe_name}.jpg"

## Process Unique Images

Download and normalize each unique supplier image only once.

Processing results are recorded separately so failed downloads or image
transformations can be inspected without losing the original product-to-image
relationships.

In [ ]:
processing_results = []

for index, row in unique_images.iterrows():
    image_url = row["image_url"]
    filename = build_normalized_filename(image_url)
    output_path = OUTPUT_DIR / filename

        # Reuse an already normalized image when the notebook is rerun.
    if output_path.exists():
        processing_results.append({
            "image_url": image_url,
            "normalized_image_path": str(output_path),
            "status": "success",
            "error": None,
        })
        continue

    try:
        response = session.get(
        image_url,
        timeout=30,
        )
        
        response.raise_for_status()

        with Image.open(BytesIO(response.content)) as source_image:
            normalized_image = normalize_product_image(source_image)

            normalized_image.save(
                output_path,
                format="JPEG",
                quality=JPEG_QUALITY,
                optimize=True,
            )

        processing_results.append({
            "image_url": image_url,
            "normalized_image_path": str(output_path),
            "status": "success",
            "error": None,
        })

    except Exception as exc:
        processing_results.append({
            "image_url": image_url,
            "normalized_image_path": None,
            "status": "failed",
            "error": str(exc),
        })

    if (index + 1) % 100 == 0:
        print(
            f"Processed {index + 1:,} / "
            f"{len(unique_images):,} images"
        )

image_processing_results = pd.DataFrame(processing_results)

print("\nProcessing complete.")
print(image_processing_results["status"].value_counts())

## Validate Processing Results

Verify that every unique source image was processed successfully before the
normalized images are used by downstream ecommerce steps.

In [ ]:
successful_images = (
    image_processing_results["status"]
    .eq("success")
    .sum()
)

failed_images = (
    image_processing_results["status"]
    .eq("failed")
    .sum()
)

assert len(image_processing_results) == len(unique_images), (
    "Not all unique images were processed."
)

assert failed_images == 0, (
    f"Image processing failures found: {failed_images}"
)

print(f"Successfully processed: {successful_images:,}")
print("Image processing validation passed.")

## Reconnect Images to Product Variants

Reconnect each normalized image to its original product and variant relationships.

This preserves the complete product-to-image mapping while ensuring that each
unique source image only needed to be processed once.

In [ ]:
normalized_image_mapping = image_processing_results[
    ["image_url", "normalized_image_path"]
].copy()

product_image_mapping = product_images.merge(
    normalized_image_mapping,
    how="left",
    on="image_url",
    validate="many_to_one",
)

assert len(product_image_mapping) == len(product_images), (
    "Image relationships changed during mapping."
)

assert product_image_mapping["normalized_image_path"].notna().all(), (
    "Some product images could not be matched to a normalized image."
)

print(f"Product-image relationships: {len(product_image_mapping):,}")
print("Normalized images successfully mapped to product variants.")

## Export Normalized Image Metadata

Export the complete product-to-image mapping for downstream ecommerce steps.

The exported metadata preserves the original product and variant relationships
while referencing the normalized local image files created by this pipeline.

In [ ]:
NORMALIZED_IMAGE_METADATA_PATH = (
    DATA_DIR / "normalized_image_metadata.csv"
)

product_image_mapping.to_csv(
    NORMALIZED_IMAGE_METADATA_PATH,
    index=False,
)

print(
    f"Exported {len(product_image_mapping):,} image relationships "
    f"to {NORMALIZED_IMAGE_METADATA_PATH}"
)

## Final Validation

Perform final integrity checks on the exported image metadata before it is used
by the downstream Shopify export pipeline.

In [ ]:
assert NORMALIZED_IMAGE_METADATA_PATH.exists(), (
    "Normalized image metadata export was not created."
)

assert len(product_image_mapping) == len(product_images), (
    "Exported image relationship count does not match the validated input."
)

assert product_image_mapping["normalized_image_path"].notna().all(), (
    "Missing normalized image paths found."
)

assert product_image_mapping["image_url"].notna().all(), (
    "Missing source image URLs found."
)

print(f"Validated image relationships: {len(product_image_mapping):,}")
print("Final image pipeline validation passed.")